In [11]:
import polars as pl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import networkx as nx

# 1. Load and merge data

In [10]:
df1=pl.read_parquet(
    'items.parquet',
    columns=['item_id','category_l1','category_l2','category_l3','category']
)
df2=pl.read_parquet(
    'transactions-202411-to-202412.parquet',
    columns=['customer_id','updated_date', 'item_id']    
)
df=df1.join(df2, on='item_id', how='left')
df=df.drop_nulls()
print(len(df))
print(df.head())

6053447
shape: (5, 7)
┌───────────────┬─────────────┬──────────────┬─────────────┬──────────┬─────────────┬──────────────┐
│ item_id       ┆ category_l1 ┆ category_l2  ┆ category_l3 ┆ category ┆ customer_id ┆ updated_date │
│ ---           ┆ ---         ┆ ---          ┆ ---         ┆ ---      ┆ ---         ┆ ---          │
│ str           ┆ str         ┆ str          ┆ str         ┆ str      ┆ i32         ┆ datetime[μs] │
╞═══════════════╪═════════════╪══════════════╪═════════════╪══════════╪═════════════╪══════════════╡
│ 0007010000886 ┆ Babycare    ┆ Bình sữa,    ┆ Núm ty      ┆ Núm ty   ┆ 6419468     ┆ 2024-12-30   │
│               ┆             ┆ phụ kiện     ┆             ┆ Pigeon   ┆             ┆ 10:22:00.050 │
│ 0007010000886 ┆ Babycare    ┆ Bình sữa,    ┆ Núm ty      ┆ Núm ty   ┆ 6695714     ┆ 2024-12-28   │
│               ┆             ┆ phụ kiện     ┆             ┆ Pigeon   ┆             ┆ 13:39:11.750 │
│ 0007010000886 ┆ Babycare    ┆ Bình sữa,    ┆ Núm ty      ┆ Núm ty  

In [26]:
print(df['category_l1'].unique())

shape: (14,)
Series: 'category_l1' [str]
[
	"Sữa nước"
	"Vệ sinh"
	"Hóa mỹ phẩm cho bé"
	"Thực phẩm cho gia đình"
	"Tã"
	…
	"Thời trang"
	"Babycare"
	"Phụ kiện"
	"Đồ chơi & Sách"
	"TPCN"
]


# 2. cat1

In [38]:
pdf = df.select(['customer_id', 'category_l1']).to_pandas()

pdf_unique = pdf.drop_duplicates()

cat_counts = pdf_unique['category_l1'].value_counts().to_dict()

# Tự ghép bảng với chính nó để sinh ra các cặp category được mua chung bởi cùng 1 customer_id
merged = pd.merge(pdf_unique, pdf_unique, on='customer_id')

pairs = merged[merged['category_l1_x'] < merged['category_l1_y']]

co_counts = pairs.groupby(['category_l1_x', 'category_l1_y']).size().reset_index(name='co_count')
def calculate_score(row):
    A = row['category_l1_x']
    B = row['category_l1_y']
    n_ab = row['co_count']
    
    if n_ab <= 1:
        return 0.0
        
    n_a = cat_counts[A] # Tổng số lần A được mua
    n_b = cat_counts[B] # Tổng số lần B được mua
    
    p_b_given_a = n_ab / n_a  # P(B|A)
    p_a_given_b = n_ab / n_b  # P(A|B)
    
    # Công thức toán học
    score = np.log10(n_ab) * (p_b_given_a + p_a_given_b)
    return score

# Áp dụng hàm tính điểm vào bảng dữ liệu
co_counts['score'] = co_counts.apply(calculate_score, axis=1)

plot_data = co_counts[co_counts['score'] > 0].sort_values(by='score', ascending=False).head(200)

print("\n--- TOP CÁC NGÀNH HÀNG THƯỜNG ĐƯỢC MUA CHUNG ---")
print(plot_data)

# --- 4. VẼ ĐỒ THỊ MẠNG LƯỚI ---
# Khởi tạo đồ thị
G = nx.Graph()

# Thêm điểm và đường nối (cạnh) với độ dày đường nối dựa trên 'score'
for _, row in plot_data.iterrows():
    G.add_edge(row['category_l1_x'], row['category_l1_y'], weight=row['score'])

plt.figure(figsize=(14, 10))

# Dàn trải các Node cho dễ nhìn (spring_layout kéo các node liên quan lại gần nhau)
pos = nx.spring_layout(G, k=1.2, seed=42) 

edges = G.edges(data=True)
weights = [edge[2]['weight'] * 15 for edge in edges] # Tùy chỉnh số 15 to/nhỏ tùy vào điểm score thực tế của bạn




--- TOP CÁC NGÀNH HÀNG THƯỜNG ĐƯỢC MUA CHUNG ---
           category_l1_x           category_l1_y  co_count     score
0               Babycare      Hóa mỹ phẩm cho bé    132920  4.788730
20    Hóa mỹ phẩm cho bé        Thực phẩm cho bé    137306  4.450599
8               Babycare        Thực phẩm cho bé    143119  4.342914
81      Thực phẩm cho bé  Thực phẩm cho gia đình     69456  4.289496
50                   Sữa        Thực phẩm cho bé    139622  4.121039
..                   ...                     ...       ...       ...
35  Hóa mỹ phẩm gia đình          Đồ chơi & Sách      6167  1.116528
39              Phụ kiện                 Textile      8907  1.070553
56              Sữa nước                 Textile     10103  0.927414
29  Hóa mỹ phẩm gia đình                 Textile      4157  0.853235
25  Hóa mỹ phẩm gia đình                Phụ kiện      3597  0.699944

[91 rows x 4 columns]


<Figure size 1400x1000 with 0 Axes>

# 3. cat2

In [36]:
pdf2 = df.select(['customer_id', 'category_l2']).to_pandas()

pdf2_unique = pdf2.drop_duplicates()

cat_counts = pdf2_unique['category_l2'].value_counts().to_dict()

# Tự ghép bảng với chính nó để sinh ra các cặp category được mua chung bởi cùng 1 customer_id
merged = pd.merge(pdf2_unique, pdf2_unique, on='customer_id')

pairs = merged[merged['category_l2_x'] < merged['category_l2_y']]

co_counts = pairs.groupby(['category_l2_x', 'category_l2_y']).size().reset_index(name='co_count')
def calculate_score(row):
    A = row['category_l2_x']
    B = row['category_l2_y']
    n_ab = row['co_count']
    
    if n_ab <= 1:
        return 0.0
        
    n_a = cat_counts[A] # Tổng số lần A được mua
    n_b = cat_counts[B] # Tổng số lần B được mua
    
    p_b_given_a = n_ab / n_a  # P(B|A)
    p_a_given_b = n_ab / n_b  # P(A|B)
    
    # Công thức toán học
    score = np.log10(n_ab) * (p_b_given_a + p_a_given_b)
    return score

# Áp dụng hàm tính điểm vào bảng dữ liệu
co_counts['score'] = co_counts.apply(calculate_score, axis=1)

plot_data = co_counts[co_counts['score'] > 0].sort_values(by='score', ascending=False).head(200)

print("\n--- TOP CÁC NGÀNH HÀNG THƯỜNG ĐƯỢC MUA CHUNG ---")
print(plot_data)

# --- 4. VẼ ĐỒ THỊ MẠNG LƯỚI ---
# Khởi tạo đồ thị
G = nx.Graph()

# Thêm điểm và đường nối (cạnh) với độ dày đường nối dựa trên 'score'
for _, row in plot_data.iterrows():
    G.add_edge(row['category_l2_x'], row['category_l2_y'], weight=row['score'])

plt.figure(figsize=(14, 10))

# Dàn trải các Node cho dễ nhìn (spring_layout kéo các node liên quan lại gần nhau)
pos = nx.spring_layout(G, k=1.2, seed=42) 

edges = G.edges(data=True)
weights = [edge[2]['weight'] * 15 for edge in edges] # Tùy chỉnh số 15 to/nhỏ tùy vào điểm score thực tế của bạn


--- TOP CÁC NGÀNH HÀNG THƯỜNG ĐƯỢC MUA CHUNG ---
             category_l2_x              category_l2_y  co_count     score
5651          Snack ăn dặm  TP từ sữa (bảo quản lạnh)     60196  3.875886
3188       Dầu ăn & Gia vị               Snack ăn dặm     50044  3.824193
3175       Dầu ăn & Gia vị        Mì & Đồ khô ăn liền     41002  3.599525
5174   Mì & Đồ khô ăn liền               Snack ăn dặm     49318  3.524906
3197       Dầu ăn & Gia vị  TP từ sữa (bảo quản lạnh)     43386  3.475332
...                    ...                        ...       ...       ...
2625  Chăm sóc sức khỏe bé  TP từ sữa (bảo quản lạnh)     14690  1.429423
227                    1Y+                    Đồ uống     13567  1.426601
204                    1Y+                TPCN cho bé     18177  1.424063
3084          Dầu sức khỏe                   Khăn ướt      8990  1.423471
2363     Chăm sóc gia đình               Snack ăn dặm      8602  1.421598

[200 rows x 4 columns]


<Figure size 1400x1000 with 0 Axes>

In [39]:
from pyvis.network import Network

net = Network(height='800px', width='100%', bgcolor='#ffffff', font_color='black', directed=False)

net.from_nx(G)

net.show_buttons(filter_=['physics'])

net.write_html('items_graph.html')
print("Đã xuất file: items_graph.html")

Đã xuất file: items_graph.html


In [37]:
from pyvis.network import Network

net = Network(height='800px', width='100%', bgcolor='#ffffff', font_color='black', notebook=False)

net.from_nx(G)

net.toggle_physics(True)

net.show_buttons(filter_=['physics'])

net.write_html('network_category_l2.html')
print("Đã tạo file network_category_l2.html thành công!")

Đã tạo file network_category_l2.html thành công!
